# 7. Networks with incomplete data

This notebook accompanies Section 7 of [*Community Detection with the Map Equation and Infomap: Theory and Applications*](https://doi.org/10.1145/3779648).

The standard map equation estimates flow from what it sees in the data. When link measurements are incomplete, those estimates drift from the true flow, and the map equation can overfit the sampling noise and report spurious modules. The regularized map equation guards against this. It places a Bayesian prior over each node's transition rates, blending the observed links with prior expectations to damp the noise.

Here you run Infomap with and without `regularized` on a network at three sampling levels and watch how regularization changes which modules survive.

In [ ]:
%matplotlib inline

from infomap import Infomap
import networkx as nx

In [ ]:
import _support as util

A synthetic benchmark network with 100 nodes, average degree 14.5, and three planted modules. You start from the complete network, then remove links to simulate incomplete observations.

In [ ]:
# First, we explore communities that Infomap detects in the network when all links are provided.

G = nx.read_edgelist("data/VII_network_complete.dat", nodetype=int)
nodesize = {node: 1.4 for node in G.nodes()}
pos = nx.spring_layout(G, seed=42)
nx.set_node_attributes(G, pos, "pos")

In [ ]:
# Infomap detects planted modules accurately.

im = Infomap(silent=True, two_level=True, num_trials=10, seed=123)
im.add_networkx_graph(G)
im.run()

modules = dict(im.modules)

util.partition(G, initial_partition=modules, no_infomap=True)
util.plot_modular_network(
    G,
    figsize=(8, 4),
    node_size=200,
    edge_width=0.5,
    node_edgecolors="white",
)

Now remove half the links at random and compare the partitions Infomap finds with and without `regularized`. The `regularization_strength` parameter scales the prior: larger values lean harder on prior expectations, smaller values trust the observed links more. This network is small, so set it to `0.7`. For larger networks (above ~300 nodes) the default of `1` works well.

In [ ]:
# The network after 50% of the links were removed.

G_50 = nx.read_edgelist("data/VII_network_incomplete_50.dat", nodetype=int)
nx.set_node_attributes(G_50, pos, "pos")

In [ ]:
# The standard map equation can overfit to the noise and detect small spurious modules.

im = Infomap(silent=True, two_level=True, num_trials=10, seed=123)
im.add_networkx_graph(G_50)
im.run()

modules = dict(im.modules)

util.partition(G_50, initial_partition=modules, no_infomap=True)
util.plot_modular_network(
    G_50,
    figsize=(8, 4),
    node_size=200,
    edge_width=0.5,
    node_edgecolors="white",
)

In [ ]:
# The regularized map equation prevents overfitting and detects planted modules.

im = Infomap(
    silent=True,
    two_level=True,
    num_trials=10,
    seed=123,
    regularized=True,
    regularization_strength=0.7,
)
im.add_networkx_graph(G_50)
im.run()

modules_regularized = dict(im.modules)

util.partition(G_50, initial_partition=modules_regularized, no_infomap=True)
util.plot_modular_network(
    G_50,
    figsize=(8, 4),
    node_size=200,
    edge_width=0.5,
    node_edgecolors="white",
)

This time remove 70% of the links. With only 30% remaining, the surviving structure carries little evidence of the planted modules, so even regularization should not invent communities that the data no longer support.

In [ ]:
# The network after 70% of the links were removed.

G_30 = nx.read_edgelist("data/VII_network_incomplete_30.dat", nodetype=int)
nx.set_node_attributes(G_30, pos, "pos")

In [ ]:
# The standard map equation overfits to the noise and detects many small spurious modules.

im = Infomap(silent=True, two_level=True, num_trials=10, seed=123)
im.add_networkx_graph(G_30)
im.run()

modules = dict(im.modules)

util.partition(G_30, initial_partition=modules, no_infomap=True)
util.plot_modular_network(
    G_30,
    figsize=(8, 4),
    node_size=200,
    edge_width=0.5,
    node_edgecolors="white",
)

In [ ]:
# The regularized map equation detects no modules if the network structure is too sparse.

im = Infomap(
    silent=True,
    two_level=True,
    num_trials=10,
    seed=123,
    regularized=True,
    regularization_strength=0.7,
)
im.add_networkx_graph(G_30)
im.run()

modules_regularized = dict(im.modules)

util.partition(G_30, initial_partition=modules_regularized, no_infomap=True)
util.plot_modular_network(
    G_30,
    figsize=(8, 4),
    node_size=200,
    edge_width=0.5,
    node_edgecolors="white",
)